In [1]:
!pip install torch matplotlib


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import itertools

torch.set_default_dtype(torch.float64)

In [6]:
def hermitian(real_part, imag_part):
    R = 0.5 * (real_part + real_part.T)        # symmetric  -> real part
    A = 0.5 * (imag_part - imag_part.T)        # antisymmetric -> imag part
    return torch.complex(R, A)
 
 
def comm(P, Q):
    return P @ Q - Q @ P
 
 
def enumerate_words(L):
    """All words over the alphabet {0:'X', 1:'Y'} of length 0..L, as tuples."""
    words = [()]
    for length in range(1, L + 1):
        words.extend(itertools.product((0, 1), repeat=length))
    return words
 
 
def parity_nontrivial(word, mu):
    """
    Parity selection rule: tr(w E_mu) (and the whole equation) vanishes
    identically unless the leading moment tr(w X_mu) has even X- and Y-counts.
    Returns False for equations that are 0=0 by symmetry (safe to drop).
    """
    cX = word.count(0) + (1 if mu == 0 else 0)
    cY = word.count(1) + (1 if mu == 1 else 0)
    return (cX % 2 == 0) and (cY % 2 == 0)
 
 
def loss(Xr, Xi, Yr, Yi, g, M, words, R=2.0, use_parity=True):
    """Sum of weighted squared loop-equation residuals (the RMA objective)."""
    X = hermitian(Xr, Xi)
    Y = hermitian(Yr, Yi)
    mats = [X, Y]
    I = torch.eye(M, dtype=torch.complex128)
 
    # equation-of-motion operators E_mu = X_mu + g [X_nu,[X_nu,X_mu]]
    E = [X + g * comm(Y, comm(Y, X)),      # E_1 (mu = 0, i.e. X)
         Y + g * comm(X, comm(X, Y))]      # E_2 (mu = 1, i.e. Y)
 
    # cache word -> matrix product (memoize prefixes/suffixes for free reuse)
    cache = {(): I}
 
    def wmat(word):
        if word not in cache:
            cache[word] = wmat(word[:-1]) @ mats[word[-1]]
        return cache[word]
 
    def ntr(Mmat):
        return torch.trace(Mmat) / M
 
    total = torch.zeros((), dtype=torch.float64)  # real scalar accumulator
 
    for word in words:
        for mu in (0, 1):
            if use_parity and not parity_nontrivial(word, mu):
                continue
            rhs = ntr(wmat(word) @ E[mu])                 # single moment term
            lhs = torch.zeros((), dtype=torch.complex128)  # splitting term
            for p, k in enumerate(word):
                if k == mu:
                    lhs = lhs + ntr(wmat(word[:p])) * ntr(wmat(word[p + 1:]))
            res = rhs - lhs
            w = R ** (-(len(word) + 1))                   # scale balancing (cf. rn)
            total = total + w * w * (res.real ** 2 + res.imag ** 2)
 
    return total
 
 
def run_rma(M, L, g, lr=0.02, num_epochs=20000, R=2.0, seed=0,
            decay_every=4000, decay_gamma=0.5):
    """Gradient descent over the entries of the two master-field matrices.
 
    Note vs the 1-matrix eigenvalue problem: optimizing full Hermitian
    matrices is a stiffer landscape, so this needs O(10^4) epochs and an
    LR schedule to drive the loss to ~1e-7 (where moments are trustworthy).
    """
    print(f"Starting RMA: M={M}, L={L}, g={g}, lr={lr}, epochs={num_epochs}")
    torch.manual_seed(seed)
 
    Xr = torch.randn(M, M, requires_grad=True)
    Xi = torch.randn(M, M, requires_grad=True)
    Yr = torch.randn(M, M, requires_grad=True)
    Yi = torch.randn(M, M, requires_grad=True)
 
    words = enumerate_words(L)
    optim = torch.optim.Adam([Xr, Xi, Yr, Yi], lr=lr)
    sched = torch.optim.lr_scheduler.StepLR(optim, step_size=decay_every,
                                            gamma=decay_gamma)
 
    for epoch in range(num_epochs):
        optim.zero_grad()
        Lval = loss(Xr, Xi, Yr, Yi, g, M, words, R=R)
        Lval.backward()
        optim.step()
        sched.step()
        if epoch % max(1, num_epochs // 10) == 0:
            print(f"  epoch {epoch}/{num_epochs}, loss {Lval.item():.6e}")
 
    with torch.no_grad():
        X = hermitian(Xr, Xi)
        Y = hermitian(Yr, Yi)
    print(f"Done. Final loss: {Lval.item():.6e}\n")
    return X, Y, Lval.item()

In [7]:
def ntr(Mmat, M):
    return (torch.trace(Mmat) / M).item()
 
 
def mpow(Mmat, n, M):
    P = torch.eye(M, dtype=torch.complex128)
    for _ in range(n):
        P = P @ Mmat
    return P
 
 
def expval_commutator_word(X, Y, p, n, m):
    """ (1/N) Tr( [X,Y]^p X^n Y^m ). Returns a complex number. """
    M = X.shape[0]
    C = comm(X, Y)
    W = mpow(C, p, M) @ mpow(X, n, M) @ mpow(Y, m, M)
    return ntr(W, M)

In [8]:
    M, L = 30, 4
 
    # --- sanity check at g = 0: X, Y independent Gaussians ---
    #     exact: tr X^2 = 1, tr X^4 = 2, tr [X,Y]^2 = -2
    X, Y, _ = run_rma(M=M, L=L, g=0.0, num_epochs=20000)
    Mn = X.shape[0]
    print("g = 0 checks (target 1, 2, -2):")
    print("  tr X^2      =", ntr(mpow(X, 2, Mn), Mn))
    print("  tr X^4      =", ntr(mpow(X, 4, Mn), Mn))
    print("  tr [X,Y]^2  =", ntr(mpow(comm(X, Y), 2, Mn), Mn))
 
    # --- interacting point ---
    g = 0.2
    X, Y, _ = run_rma(M=M, L=L, g=g, num_epochs=20000)
    Mn = X.shape[0]
    print(f"g = {g}:")
    print("  tr X^2          =", ntr(mpow(X, 2, Mn), Mn))
    print("  tr [X,Y]^2      =", ntr(mpow(comm(X, Y), 2, Mn), Mn))
    print("  tr [X,Y]^2 X^2 Y^2 =", expval_commutator_word(X, Y, p=2, n=2, m=2))

Starting RMA: M=30, L=4, g=0.0, lr=0.02, epochs=20000
  epoch 0/20000, loss 3.165010e+04
  epoch 2000/20000, loss 5.332264e-01
  epoch 4000/20000, loss 5.919513e-02
  epoch 6000/20000, loss 1.463309e-02
  epoch 8000/20000, loss 1.539827e-03
  epoch 10000/20000, loss 2.550249e-04
  epoch 12000/20000, loss 5.822789e-06
  epoch 14000/20000, loss 4.672347e-08
  epoch 16000/20000, loss 6.172423e-09
  epoch 18000/20000, loss 6.094661e-09
Done. Final loss: 5.888072e-09

g = 0 checks (target 1, 2, -2):
  tr X^2      = (0.9999999512864198-1.3434840121549432e-18j)
  tr X^4      = (2.0001854880354366-5.415768874840087e-18j)
  tr [X,Y]^2  = (-1.9978924295449296+2.4838031889054086e-18j)
Starting RMA: M=30, L=4, g=0.2, lr=0.02, epochs=20000
  epoch 0/20000, loss 8.257724e+06
  epoch 2000/20000, loss 3.209338e+02
  epoch 4000/20000, loss 6.734063e+01
  epoch 6000/20000, loss 2.971796e+01
  epoch 8000/20000, loss 1.070961e+01
  epoch 10000/20000, loss 5.351877e+00
  epoch 12000/20000, loss 2.056192e+0